# Supervised autoencoder on the T = 10 benign (all 8 folders) + GridSybil windows (all/T10_benign_grid)

The protocol of `task/autoencoder_st_t32` at T = 10: the depth-configurable ConvAutoencoder of
`task/autoencoder_benign_t20` (input 14 × 10 = 13 features + mask, z = 16, 1 stride-2 stage(s): encoder strides 1, 2, 1; decoder ConvTranspose k3 s1 (length kept), k4 s2;
52,861 parameters) with the same small **classifier head on z** (Linear 16→32 → ReLU → Dropout 0.2 → Linear 32→1,
577 parameters; 53,438 in total). Loss = class-weighted BCE (`pos_weight` = n_benign / n_gridsybil of the
**train** split) + λ · masked MSE on real rows (λ = `recon_weight` = 1). Data:
`src/data/encoder_input/all/T10_benign_grid/`: benign windows (sender never an attacker) from **all 8 scenario
folders** + GridSybil attack windows from the 2 GridSybil folders, with train / val / test = **85 / 7.5 / 7.5 by
vehicle key** `<group>:<vehicle>` fixed on disk (the all/T32_benign_grid split). Every training setting equals the
T = 32 package's.

Labels are used in training (supervised). The val split drives the LR schedule, early stopping, the best checkpoint
and both alarm thresholds (max F1 on val; benign FPR 5% on val). The test split is never read by `train.py`; here it
is read in section 4, and its labels only after every test window has been scored.

This notebook runs from the repo root, imports the package classes and reads the run folder written by
`npm run aest10:train`. Sections: 1 input, 2 model, 3 training record, 4 test, 5 comparison, 6 analysis. Not the
thesis TimesNet model.

Setup: repo-root check, package imports, the run folder (the newest `runs/*/summary.json`).

In [ ]:
import json
import re
from pathlib import Path

assert Path("task/autoencoder_st_t10/config.py").is_file(), "run this notebook from the repo root"

from task.autoencoder_st_t10.config import SupervisedAEConfig, TOTAL_PARAMS
from task.autoencoder_st_t10.data import WindowDataset
from task.autoencoder_st_t10.metrics import TrackScores, ranking_metrics, threshold_metrics, val_thresholds
from task.autoencoder_st_t10.model import SupervisedAutoencoder, count_params
from task.autoencoder_st_t10.trainer import Trainer

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import precision_recall_curve, roc_auc_score, roc_curve

settings = SupervisedAEConfig()
runs = sorted(Path(settings.runs_dir).glob("*/summary.json"), key=lambda p: p.stat().st_mtime)
assert runs, "no finished run: npm run aest10:train"
RUN = runs[-1].parent
META = json.loads((Path(settings.data_dir) / "metadata.json").read_text())
FEATURES = META["features"]
print("run folder:", RUN)
print("data:", settings.data_dir)

## 1. Input

The three splits as written on disk. Vehicle keys never cross splits (checked below). Class counts per split from the
data for train and val; test labels are not read in this section (its class counts are shown from `metadata.json`
only, as written by the pipeline). Every track (one link = one receiver hearing one sender pseudonym) has one label
(checked on train / val).

In [ ]:
load = dict(workers=settings.workers, cache_dir=settings.cache_dir)
splits = {name: WindowDataset.load(settings.data_dir, name, **load) for name in ("train", "val", "test")}
train, val, test = splits["train"], splits["val"], splits["test"]
veh = {name: set(d.vehicles) for name, d in splits.items()}
assert not (veh["train"] & veh["val"]) and not (veh["train"] & veh["test"]) and not (veh["val"] & veh["test"])
tracks = {name: TrackScores(d.tracks) for name, d in splits.items()}
rows = []
for name in ("train", "val"):
    d = splits[name]
    for cls, v in (("Benign", 0), ("GridSybil", 1)):
        s = d.y == v
        rows.append(
            {
                "split": name,
                "class": cls,
                "windows": int(s.sum()),
                "messages": int(d.n_messages[s].sum()),
                "tracks": len(set(d.tracks[s])),
                "vehicle keys": len(set(d.vehicles[s])),
            }
        )
counts = pd.DataFrame(rows).pivot(index="split", columns="class").loc[["train", "val"]]
counts[("windows", "total")] = counts["windows"].sum(axis=1)
print("no vehicle key in two splits:", True)
print("tracks with mixed labels:", {name: tracks[name].track_labels(splits[name].y)[1] for name in ("train", "val")})
print(
    f"test (labels not read here): {len(test):,} windows, {len(tracks['test']):,} tracks, "
    f"{len(veh['test']):,} vehicle keys"
)
counts

Split rule and per-split class counts as recorded by the pipeline in `metadata.json` (test included).

In [ ]:
print(json.dumps({k: v for k, v in META.get("split", {}).items() if k != "vehicle_ids"}, indent=1))
for key in ("counts_by_split", "counts_by_split_class", "class_balance"):
    if key in META:
        print(f"\n{key}:")
        print(json.dumps(META[key], indent=1)[:2000])

Split diagram: windows per split (benign + GridSybil for train / val; test as one bar), with the vehicle keys behind each split.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 2.8))
top = len(train) * 1.35
ax.barh(0, len(test), color="tab:grey", label="test (labels read in section 4)")
ax.text(len(test) + top * 0.01, 0, f"{len(test):,} windows, {len(veh['test']):,} vehicle keys", va="center", fontsize=9)
for y, name in ((1, "val"), (2, "train")):
    b, g = counts.loc[name, ("windows", "Benign")], counts.loc[name, ("windows", "GridSybil")]
    ax.barh(y, b, color="tab:blue", label="Benign" if y == 1 else None)
    ax.barh(y, g, left=b, color="tab:orange", label="GridSybil" if y == 1 else None)
    ax.text(
        b + g + top * 0.01, y, f"{b:,} + {g:,} = {b + g:,} windows, {len(veh[name]):,} keys", va="center", fontsize=9
    )
ax.set_yticks(range(3), ["test", "val", "train"])
ax.set_xlim(0, top)
ax.set_xlabel("windows")
ax.legend(loc="lower right", fontsize=8)
ax.set_title("Train / val / test = 85 / 7.5 / 7.5 by vehicle key (fixed on disk)")
plt.show()

Train / val windows per scenario folder and class: benign windows come from every folder, GridSybil only from the two
GridSybil folders. The same benign traffic recurs in the four families of one period (F1), so the benign set holds
about four copies of each period's traffic.

In [ ]:
fam = (
    pd.concat(
        [
            pd.crosstab(np.asarray(d.info["scenario"]), np.where(d.y == 1, "GridSybil", "Benign")).add_prefix(f"{n} ")
            for n, d in (("train", train), ("val", val))
        ],
        axis=1,
    )
    .fillna(0)
    .astype(int)
)
fam.loc["total"] = fam.sum()
fam

Shape and padding: each window is up to 10 messages (1 Hz) × 13 normalised features, padding rows zero, the mask
marks real rows; the model gets the mask as a 14th channel (**14 × 10**). Padding and the share of 1–3-row
windows per split (per class for train / val). The class weight of the BCE comes from the train counts.

In [ ]:
print(
    "window x:",
    train.x.shape[1:],
    "+ mask:",
    train.mask.shape[1:],
    "-> model input:",
    (settings.n_features + 1, settings.seq_len),
)
rows = []
for name, d in splits.items():
    groups = [("all", np.ones(len(d), bool))]
    if name != "test":
        groups += [("Benign", d.y == 0), ("GridSybil", d.y == 1)]
    for cls, s in groups:
        n = d.n_messages[s]
        rows.append(
            {
                "split": name,
                "class": cls,
                "windows": int(s.sum()),
                "median rows": np.median(n),
                "mean rows": round(n.mean(), 2),
                f"share = {settings.seq_len}": round(float((n == settings.seq_len).mean()), 3),
                "share 1-3 rows": round(float((n <= 3).mean()), 3),
                "padding share": round(1 - d.mask[s].mean(), 3),
            }
        )
display(pd.DataFrame(rows).set_index(["split", "class"]))
n_b, n_g = int((train.y == 0).sum()), int((train.y == 1).sum())
pos_weight = n_b / n_g
summary = json.loads((RUN / "summary.json").read_text())
print(
    f"pos_weight = n_benign / n_gridsybil (train) = {n_b:,} / {n_g:,} = {pos_weight:.4f}  "
    f"(run: {summary['pos_weight']:.4f}); GridSybil share of train {n_g / (n_b + n_g):.3f}"
)
assert np.isclose(pos_weight, summary["pos_weight"])

fig, ax = plt.subplots(figsize=(8, 3.5))
for v, name in ((0, "Benign"), (1, "GridSybil")):
    ax.hist(train.n_messages[train.y == v], bins=np.arange(1, settings.seq_len + 2) - 0.5, alpha=0.6, label=name)
ax.set_xlabel("real messages per window")
ax.set_ylabel("train windows")
ax.set_yscale("log")
ax.legend()
ax.set_title("Train: real messages per window by class")
plt.show()

## 2. Model

Layers with their output shapes for one batch of 2 windows, including the classifier head, and the trainable
parameter count (asserted against 53,438 = 52,861 + 577; 1 stride-2 stage(s): encoder strides 1, 2, 1; decoder ConvTranspose k3 s1 (length kept), k4 s2).

In [ ]:
model = SupervisedAutoencoder(settings)
x0, m0 = torch.zeros(2, settings.seq_len, settings.n_features), torch.ones(2, settings.seq_len, dtype=torch.bool)
table = []
layer_types = (torch.nn.Conv1d, torch.nn.ConvTranspose1d, torch.nn.Linear, torch.nn.Dropout)
hooks = [
    mod.register_forward_hook(
        lambda mod, inp, out, name=name: table.append(
            (name, type(mod).__name__, tuple(out.shape[1:]), sum(p.numel() for p in mod.parameters()))
        )
    )
    for name, mod in model.named_modules()
    if isinstance(mod, layer_types)
]
model.eval()
with torch.no_grad():
    recon0, z0, logit0 = model(x0, m0)
for h in hooks:
    h.remove()
display(pd.DataFrame(table, columns=["layer", "type", "output (per window)", "params"]))
print("reconstruction:", tuple(recon0.shape[1:]), " z:", tuple(z0.shape[1:]), " logit:", tuple(logit0.shape[1:]))
print(
    f"trainable parameters: {count_params(model):,} = autoencoder {count_params(model.autoencoder):,} "
    f"+ head {count_params(model.head):,}"
)
assert count_params(model) == TOTAL_PARAMS == summary["params"] == 53438

## 3. Training record

What `task.autoencoder_st_t10.train` wrote: the summary (without the threshold tables), the per-epoch history
re-plotted (losses train vs val; val AUROC / PR-AUC / F1 / balanced accuracy; LR and gradient norm), and log lines.
Train metrics are computed from the running epoch outputs (train mode: dropout on, weights changing within the epoch),
so train and val metrics are not strictly comparable.

In [ ]:
print(json.dumps({k: v for k, v in summary.items() if k not in ("thresholds", "best_val", "family_counts")}, indent=2))
print("best epoch val metrics:", {k: round(v, 4) for k, v in summary["best_val"].items()})
history = pd.read_json(RUN / "history.jsonl", lines=True)
best = summary["best_epoch"]
print("epoch 1 val AUROC:", round(float(history.val_auroc.iloc[0]), 4))
fig, axes = plt.subplots(3, 1, figsize=(9, 10), sharex=True)
for key, c in (("loss", "tab:blue"), ("bce", "tab:orange"), ("recon", "tab:green")):
    axes[0].plot(history.epoch, history[f"train_{key}"], color=c, ls="--", label=f"train {key}")
    axes[0].plot(history.epoch, history[f"val_{key}"], color=c, label=f"val {key}")
axes[0].set_yscale("log")
axes[0].set_ylabel("loss")
axes[0].legend(fontsize=8, ncol=3)
axes[0].set_title(f"{RUN.name}: per epoch ({summary['stop_reason']}, best epoch {best})")
for key in ("auroc", "pr_auc", "f1", "balanced_accuracy"):
    axes[1].plot(history.epoch, history[f"val_{key}"], label=f"val {key}")
axes[1].plot(history.epoch, history.train_auroc, color="grey", ls="--", label="train auroc (running)")
axes[1].set_ylabel("metric")
axes[1].legend(fontsize=8)
axes[2].plot(history.epoch, history.grad_norm_mean, color="tab:green", label="grad norm mean (pre-clip)")
axes[2].plot(history.epoch, history.grad_norm_max, color="tab:olive", ls=":", label="grad norm max")
axes[2].set_ylabel("gradient L2 norm")
axes[2].set_xlabel("epoch")
lr_ax = axes[2].twinx()
lr_ax.plot(history.epoch, history.lr, color="tab:purple", label="LR")
lr_ax.set_yscale("log")
lr_ax.set_ylabel("learning rate")
for a in axes:
    for c in summary["lr_changes"]:
        a.axvline(c["after_epoch"] + 0.5, color="grey", ls=":", lw=1)
    a.axvline(best, color="red", ls="--", lw=1)
h1, l1 = axes[2].get_legend_handles_labels()
h2, l2 = lr_ax.get_legend_handles_labels()
axes[2].legend(h1 + h2, l1 + l2, fontsize=8)
fig.tight_layout()
plt.show()

log = (RUN / "train.log").read_text().splitlines()
ep = [l for l in log if re.search(r" epoch +\d+ ", l)]
head = [l for l in log if " epoch " not in l and "lr reduced" not in l][:5]
print(
    "\n".join(
        head
        + ["..."]
        + ep[:3]
        + ["..."]
        + [l for l in ep if "*best" in l][-2:]
        + ["..."]
        + [l for l in log if "lr reduced" in l]
        + ["..."]
        + [l for l in log if "early stopping:" in l]
    )
)

The two alarm thresholds, both picked on the **val** split by `train.py` (window level, and per track on the max /
mean of the window probabilities): max F1 on val, and the 95th percentile of the benign val probabilities (benign
FPR 5% on val). Recomputed here from `best.pt` on val and compared with the summary.

In [ ]:
trainer = Trainer.from_checkpoint(RUN / "best.pt")
cfg = trainer.settings
val_out = trainer.predict(val.x, val.mask)
thr_win = val_thresholds(val.y, val_out["prob"], cfg.target_fpr)
same = all(np.isclose(thr_win[k]["threshold"], summary["thresholds"]["window"][k]["threshold"]) for k in thr_win)
print(
    "val AUROC of best.pt:",
    round(ranking_metrics(val.y, val_out["prob"])["auroc"], 5),
    "(run:",
    round(summary["best_val_auroc"], 5),
    ")",
)
print("window thresholds equal the run's summary:", same)
THR = summary["thresholds"]
pd.DataFrame(
    {
        k: {
            "threshold": v["threshold"],
            "val precision": v["precision"],
            "val recall": v["recall"],
            "val F1": v["f1"],
            "val benign FPR": v["benign_fpr"],
        }
        for k, v in THR["window"].items()
    }
).T.round(4)

## 4. Test

**Scoring first, labels after.** Every test window gets its GridSybil probability, reconstruction error and z from
`best.pt` (inputs: x and mask only). Per-track scores are the max and mean of the window probabilities.

In [ ]:
test_out = trainer.predict(test.x, test.mask)
test_track = tracks["test"].aggregate(test_out["prob"])
print(
    f"scored {len(test):,} test windows in {len(tracks['test']):,} tracks; prob range "
    f"[{test_out['prob'].min():.4f}, {test_out['prob'].max():.4f}]"
)

Labels are read now (GridSybil = positive). Track label = majority of its windows; mixed tracks are counted.

In [ ]:
y_win = test.y.astype(int)
y_trk, mixed = tracks["test"].track_labels(y_win)
print(f"tracks whose windows disagree on the label: {mixed}")
print(f"windows: benign {int((y_win == 0).sum()):,}, GridSybil {int(y_win.sum()):,} (share {y_win.mean():.3f})")
print(f"tracks:  benign {int((y_trk == 0).sum()):,}, GridSybil {int(y_trk.sum()):,} (share {y_trk.mean():.3f})")

AUROC, PR-AUC (chance = GridSybil share) and precision / recall / F1 / balanced accuracy / benign FPR at the two val
thresholds, per window and per track (max / mean). Also the plain 0.5 threshold, for reference.

In [ ]:
levels = {
    "window": (y_win, test_out["prob"], THR["window"]),
    "track max": (y_trk, test_track["max"], THR["track"]["max"]),
    "track mean": (y_trk, test_track["mean"], THR["track"]["mean"]),
}
rows = []
for level, (y, s, thr) in levels.items():
    base = {"level": level, **ranking_metrics(y, s), "chance": y.mean()}
    picks = {**{k: v["threshold"] for k, v in thr.items()}, "fixed 0.5": 0.5}
    for name, t in picks.items():
        m = threshold_metrics(y, s, t)
        rows.append(
            {
                **base,
                "threshold": name,
                "θ": t,
                **{k: m[k] for k in ("precision", "recall", "f1", "balanced_accuracy", "benign_fpr")},
                **m["confusion"],
            }
        )
res = pd.DataFrame(rows)
display(res.drop(columns=["tn", "fp", "fn", "tp"]).set_index(["level", "threshold"]).round(4))

Confusion matrices per window at the two val thresholds (rows = truth, columns = prediction).

In [ ]:
for _, r in res[(res.level == "window") & (res.threshold != "fixed 0.5")].iterrows():
    cm = pd.DataFrame(
        [[r.tn, r.fp], [r.fn, r.tp]], index=["true benign", "true GridSybil"], columns=["pred benign", "pred GridSybil"]
    )
    print(f"{r.threshold} (θ = {r['θ']:.4f})")
    display(cm)

ROC and precision-recall curves (window and track max), with the two val thresholds marked on the window curves.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5))
for level in ("window", "track max"):
    y, s, _ = levels[level]
    fpr, tpr, _ = roc_curve(y, s)
    p, r, _ = precision_recall_curve(y, s)
    a1.plot(fpr, tpr, label=f"{level} (AUROC {roc_auc_score(y, s):.3f})")
    a2.plot(r, p, label=level)
for name, v in THR["window"].items():
    m = threshold_metrics(y_win, test_out["prob"], v["threshold"])
    a1.plot(m["benign_fpr"], m["recall"], "o", label=f"window @ {name}")
    a2.plot(m["recall"], m["precision"], "o", label=f"window @ {name}")
a1.plot([0, 1], [0, 1], "k:", lw=1)
a1.set_xlabel("benign FPR")
a1.set_ylabel("recall (TPR)")
a1.legend(fontsize=8)
a2.axhline(y_win.mean(), color="k", ls=":", lw=1, label="chance")
a2.set_xlabel("recall")
a2.set_ylabel("precision")
a2.legend(fontsize=8)
a1.set_title("ROC (test)")
a2.set_title("Precision-recall (test)")
fig.tight_layout()
plt.show()

Test probability histogram by class, with the two val thresholds.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
bins = np.linspace(0, 1, 51)
ax.hist(test_out["prob"][y_win == 0], bins=bins, alpha=0.6, density=True, label="benign")
ax.hist(test_out["prob"][y_win == 1], bins=bins, alpha=0.6, density=True, label="GridSybil")
for (name, v), ls in zip(THR["window"].items(), ("--", ":")):
    ax.axvline(v["threshold"], color="black", ls=ls, lw=1, label=f"val {name} ({v['threshold']:.3f})")
ax.set_yscale("log")
ax.set_xlabel("P(GridSybil)")
ax.set_ylabel("density")
ax.legend(fontsize=8)
ax.set_title("Test windows: predicted probability by class")
plt.show()

Length shortcut check: test AUROC by window length (real messages), the length-only AUROC (best direction) for
reference, and the AUROC on windows with at least 4 real rows.

In [ ]:
def length_auroc(y, n):
    a = roc_auc_score(y, n)
    return max(a, 1 - a)


T = settings.seq_len
groups = [(1, 3), (4, T - 1), (T, T), (4, T)]
rows = []
for lo, hi in groups:
    s = (test.n_messages >= lo) & (test.n_messages <= hi)
    y = y_win[s]
    if len(set(y.tolist())) < 2:
        continue
    rows.append(
        {
            "real rows": f"{lo}-{hi}",
            "windows": int(s.sum()),
            "GridSybil share": round(y.mean(), 3),
            "AUROC": roc_auc_score(y, test_out["prob"][s]),
            "length-only AUROC (best direction)": length_auroc(y, test.n_messages[s]) if lo != hi else float("nan"),
        }
    )
print("length-only AUROC, all test windows (best direction):", round(length_auroc(y_win, test.n_messages), 4))
by_length = pd.DataFrame(rows).set_index("real rows").round(4)
by_length

**Benign windows by scenario folder.** The benign traffic of the DataReplay / DoS folders never appears with a
GridSybil label, so: do benign windows from those folders get false alarms more often than benign windows from the
GridSybil folders? Per folder: benign test windows, mean P(GridSybil), benign FPR at both val thresholds, and the
AUROC of the test GridSybil windows of the same period against that folder's benign windows.

In [ ]:
scen = np.asarray(test.info["scenario"])
period = np.asarray([s.rsplit("_", 1)[-1] for s in scen])
rows = []
for sc in sorted(set(scen[y_win == 0])):
    b = (scen == sc) & (y_win == 0)
    g = (period == sc.rsplit("_", 1)[-1]) & (y_win == 1)
    row = {
        "folder": sc,
        "benign test windows": int(b.sum()),
        "share 1-3 rows": round(float((test.n_messages[b] <= 3).mean()), 3),
        "mean P(GridSybil)": test_out["prob"][b].mean(),
    }
    for name, v in THR["window"].items():
        row[f"benign FPR @ {name}"] = float((test_out["prob"][b] >= v["threshold"]).mean())
    sb, sg = test_out["prob"][b], test_out["prob"][g]
    row["AUROC vs same-period GridSybil"] = roc_auc_score(np.r_[np.zeros(len(sb)), np.ones(len(sg))], np.r_[sb, sg])
    rows.append(row)
by_folder = pd.DataFrame(rows).set_index("folder").round(4)
by_folder

**Per period** (0709 vs 1416): all test windows of one period (benign from its four folders + its GridSybil windows).
GridSybil_0709 is the 2025 re-simulation of the 0709 traffic (F14).

In [ ]:
rows = []
for p in ("0709", "1416"):
    s = period == p
    y, sc = y_win[s], test_out["prob"][s]
    row = {"period": p, "windows": int(s.sum()), "GridSybil share": round(y.mean(), 3), **ranking_metrics(y, sc)}
    for name, v in THR["window"].items():
        m = threshold_metrics(y, sc, v["threshold"])
        row[f"F1 @ {name}"] = m["f1"]
        row[f"recall @ {name}"] = m["recall"]
        row[f"benign FPR @ {name}"] = m["benign_fpr"]
    rows.append(row)
by_period = pd.DataFrame(rows).set_index("period").round(4)
by_period

Mean per-feature reconstruction error by class on the test windows, and the per-feature AUROC of that error.

In [ ]:
ef = test_out["error_feature"]
print(
    f"mean window reconstruction error: benign {test_out['error'][y_win == 0].mean():.4f}, "
    f"GridSybil {test_out['error'][y_win == 1].mean():.4f}; "
    f"AUROC of the error {roc_auc_score(y_win, test_out['error']):.4f}"
)
pd.DataFrame(
    {
        "benign": ef[y_win == 0].mean(0),
        "GridSybil": ef[y_win == 1].mean(0),
        "AUROC": [roc_auc_score(y_win, ef[:, j]) for j in range(ef.shape[1])],
    },
    index=FEATURES,
).round(4).sort_values("AUROC", ascending=False)

## 5. Comparison (context only)

Other numbers on GridSybil from this repo. The setups differ (T, data cut, split, metric unit, seeds), so these rows
are context, not a controlled comparison: `task/autoencoder_st_t32` (the same protocol at T = 32, same split, mean of
seeds 0–3 from agent.md), `task/autoencoder_st` (supervised, T48_tvt, window AUROC), and the VeReMi-Extension paper's
plausibility detector re-implemented in `src/baselines/plausibility/` (CaTCh checks, **per received message F1**, not
a window AUROC).

In [ ]:
w = res[(res.level == "window")].iloc[0]
w_f1 = res[(res.level == "window") & (res.threshold == "max_f1")]
cmp = [
    {
        "run": f"this run ({RUN.name}): supervised AE + head, all/T10_benign_grid",
        "unit": "window",
        "metric": "AUROC / F1 @ val max-F1",
        "value": f"{w.auroc:.3f} / {float(w_f1.f1.iloc[0]):.3f}" if len(w_f1) else f"{w.auroc:.3f}",
        "seeds": 1,
    },
    {
        "run": "task/autoencoder_st_t32 seeds 0-3: supervised AE + head, all/T32_benign_grid",
        "unit": "window",
        "metric": "AUROC (mean ± sd)",
        "value": "0.9289 ± 0.0018",
        "seeds": 4,
    },
    {
        "run": "task/autoencoder_st seed0: supervised AE + head, T48_tvt",
        "unit": "window",
        "metric": "AUROC",
        "value": "0.929",
        "seeds": 1,
    },
    {
        "run": "plausibility baseline (paper Table III, CaTCh), GridSybil 0709 / 1416",
        "unit": "received message",
        "metric": "F1",
        "value": "0.727 / 0.692",
        "seeds": "-",
    },
]
pd.DataFrame(cmp).set_index("run")

The numbers the analysis below refers to.

In [ ]:
results = {
    "run": RUN.name,
    "params": summary["params"],
    "best_epoch": summary["best_epoch"],
    "best_val_auroc": round(summary["best_val_auroc"], 4),
    "epoch1_val_auroc": round(float(history.val_auroc.iloc[0]), 4),
    "epochs_run": summary["epochs_run"],
    "stop_reason": summary["stop_reason"],
    "lr_changes": len(summary["lr_changes"]),
    "train_minutes": round(summary["train_seconds"] / 60, 1),
    "pos_weight": round(summary["pos_weight"], 4),
    "mixed_label_tracks": mixed,
    "test_windows": len(test),
    "test_tracks": len(tracks["test"]),
    "test_gridsybil_share": round(float(y_win.mean()), 4),
    "length_only_auroc": round(length_auroc(y_win, test.n_messages), 4),
    "reconstruction_error_auroc": round(roc_auc_score(y_win, test_out["error"]), 4),
}
for _, r in res.iterrows():
    results[f"{r.level}/{r.threshold}"] = {
        k: round(float(r[k]), 4)
        for k in ("auroc", "pr_auc", "θ", "precision", "recall", "f1", "balanced_accuracy", "benign_fpr")
    }
results

## 6. Analysis

**Not written yet.** This notebook was copied from the T = 32 notebook before any T = 10 run existed; the T = 32
analysis text was removed because its numbers do not describe this run (RULE 1). Write the analysis from the
outputs above after `npm run aest10:train` has finished (one seed per run folder, no confidence interval; caveats F1, F11, F13,
F14 as in the T = 32 notebook).